# 04 - Giao thức Conv1D cho CDC Diabetes

Notebook này chuẩn bị workflow phân loại tabular CDC Diabetes cho Assignment 05. Nó phân tích dataset, tạo split deterministic, chọn preprocessing và siêu tham số bằng BasicCNN1D trên validation, rồi so sánh bốn kiến trúc Conv1D ở phần cuối.


## 1. Định nghĩa bài toán

Bài toán là phân loại ba lớp của `Diabetes_012`: No diabetes, Prediabetes, và Diabetes. Assignment yêu cầu family mô hình CNN, nên dataset tabular được biểu diễn như sequence ngắn để dùng Conv1D.


## Giới hạn mô hình hóa quan trọng

CNN tự nhiên khai thác cấu trúc không gian hoặc tuần tự cục bộ. 21 predictor của CDC Diabetes là biến tabular, và các cột đứng cạnh nhau không hàm ý locality vật lý như pixel kề nhau. Vì vậy Conv1D ở đây là adaptation theo yêu cầu assignment. Notebook không khẳng định CNN là tự nhiên hoặc tối ưu về lý thuyết cho dữ liệu này, và giữ nguyên thứ tự feature trong dataset thay vì sắp xếp lại cột để tạo locality nhân tạo.


In [1]:
from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from models.architectures import build_basic_cnn_1d

SEED = 42
EXPECTED_PYTHON = "C:/Users/anhca/anaconda3/envs/tf312/python.exe"
actual_python = sys.executable.replace("\\", "/")
assert actual_python.lower() == EXPECTED_PYTHON.lower(), actual_python

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "AGENTS.md").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("A05 project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "datasets" / "diabetes" / "diabetes_012_health_indicators_BRFSS2015.csv"
SPLIT_DIR = PROJECT_ROOT / "results" / "splits"
HP_DIR = PROJECT_ROOT / "results" / "hyperparameters"
FIG_DIR = PROJECT_ROOT / "results" / "figures" / "diabetes" / "hyperparameters"
for path in [SPLIT_DIR, HP_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing required diabetes dataset: {DATA_PATH.relative_to(PROJECT_ROOT).as_posix()}")

print({
    "python_executable": actual_python,
    "python_version": sys.version.split()[0],
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "devices": [str(device) for device in tf.config.list_physical_devices()],
    "project_root": PROJECT_ROOT.name,
})


{'python_executable': 'C:/Users/anhca/anaconda3/envs/tf312/python.exe', 'python_version': '3.12.14', 'tensorflow_version': '2.21.0', 'keras_version': '3.15.1', 'devices': ["PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')"], 'project_root': 'A05'}


## 2. Mô tả dataset

Nguồn dữ liệu duy nhất được dùng là `datasets/diabetes/diabetes_012_health_indicators_BRFSS2015.csv`. Raw CSV chỉ được đọc và không bao giờ bị sửa.


In [2]:
df = pd.read_csv(DATA_PATH)
df.insert(0, "row_index", np.arange(len(df), dtype=np.int64))

TARGET = "Diabetes_012"
feature_columns = [column for column in df.columns if column not in ["row_index", TARGET]]

assert len(df) == 253680
assert TARGET in df.columns
assert len(feature_columns) == 21

target_distribution = df[TARGET].value_counts().sort_index().rename_axis("target").reset_index(name="count")
target_distribution["proportion"] = target_distribution["count"] / len(df)
print({"rows": len(df), "columns": len(df.columns) - 1, "predictor_features": len(feature_columns)})
target_distribution


{'rows': 253680, 'columns': 22, 'predictor_features': 21}


   target   count  proportion
0     0.0  213703    0.842412
1     1.0    4631    0.018255
2     2.0   35346    0.139333

## 3. Ý nghĩa feature

Bảng dưới đây ghi lại thứ tự predictor dùng cho biểu diễn Conv1D. Thứ tự đến từ CSV và được giữ nguyên cho thí nghiệm chính.


In [3]:
feature_table = pd.DataFrame({
    "position": np.arange(1, len(feature_columns) + 1),
    "feature": feature_columns,
})
feature_table


    position               feature
0          1                HighBP
1          2              HighChol
2          3             CholCheck
3          4                   BMI
4          5                Smoker
5          6                Stroke
6          7  HeartDiseaseorAttack
7          8          PhysActivity
8          9                Fruits
9         10               Veggies
10        11     HvyAlcoholConsump
11        12         AnyHealthcare
12        13           NoDocbcCost
13        14               GenHlth
14        15              MentHlth
15        16              PhysHlth
16        17              DiffWalk
17        18                   Sex
18        19                   Age
19        20             Education
20        21                Income

## 4. Kiểu dữ liệu

Data type được kiểm tra từ CSV thật. Phần lớn biến là indicator rời rạc hoặc trường khảo sát ordinal, trong khi BMI có range số rộng hơn.


## 5. Giá trị thiếu

Missing values được kiểm tra trước khi split hoặc preprocessing. Không thêm imputation nếu dữ liệu không yêu cầu.


In [4]:
dtype_table = (
    df[feature_columns + [TARGET]]
    .dtypes
    .astype(str)
    .reset_index()
    .rename(columns={"index": "column", 0: "dtype"})
)
missing_table = (
    df[feature_columns + [TARGET]]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_values"})
)
dtype_missing_table = dtype_table.merge(missing_table, on="column")
print({"total_missing_values": int(dtype_missing_table["missing_values"].sum())})
dtype_missing_table


{'total_missing_values': 0}


                  column    dtype  missing_values
0                 HighBP  float64               0
1               HighChol  float64               0
2              CholCheck  float64               0
3                    BMI  float64               0
4                 Smoker  float64               0
5                 Stroke  float64               0
6   HeartDiseaseorAttack  float64               0
7           PhysActivity  float64               0
8                 Fruits  float64               0
9                Veggies  float64               0
10     HvyAlcoholConsump  float64               0
11         AnyHealthcare  float64               0
12           NoDocbcCost  float64               0
13               GenHlth  float64               0
14              MentHlth  float64               0
15              PhysHlth  float64               0
16              DiffWalk  float64               0
17                   Sex  float64               0
18                   Age  float64               0


## 6. Phân tích duplicate

Audit phát hiện các dòng duplicate. Chúng không tự động bị xem là lỗi dữ liệu và không bị drop. Vì dataset có nhiều biến khảo sát rời rạc hoặc binary và không có respondent identifier, các dòng giống hệt có thể đại diện cho những người khác nhau có đặc trưng ghi nhận giống nhau. Workflow chính giữ đủ 253,680 dòng. Một giới hạn là các feature vector giống hệt có thể xuất hiện giữa train và test vì không có respondent ID để grouped split.


In [5]:
exact_duplicate_extra_rows = int(df[feature_columns + [TARGET]].duplicated().sum())
exact_duplicate_rows_involved = int(df[df[feature_columns + [TARGET]].duplicated(keep=False)].shape[0])
same_feature_target_group_sizes = df.groupby(feature_columns + [TARGET], dropna=False).size()
same_feature_same_target_groups = int((same_feature_target_group_sizes > 1).sum())
same_feature_same_target_extra_rows = int((same_feature_target_group_sizes[same_feature_target_group_sizes > 1] - 1).sum())

feature_target_counts = df.groupby(feature_columns, dropna=False)[TARGET].nunique()
conflicting_feature_groups = feature_target_counts[feature_target_counts > 1]
conflicting_feature_group_count = int(len(conflicting_feature_groups))
conflicting_feature_rows = int(
    df.merge(conflicting_feature_groups.rename("target_count").reset_index()[feature_columns], on=feature_columns, how="inner").shape[0]
)

duplicate_summary = pd.DataFrame([
    {
        "category": "A_exact_duplicates_all_predictors_plus_target",
        "groups": same_feature_same_target_groups,
        "extra_rows": exact_duplicate_extra_rows,
        "rows_involved": exact_duplicate_rows_involved,
    },
    {
        "category": "B_identical_21_features_same_target",
        "groups": same_feature_same_target_groups,
        "extra_rows": same_feature_same_target_extra_rows,
        "rows_involved": exact_duplicate_rows_involved,
    },
    {
        "category": "C_identical_21_features_different_targets",
        "groups": conflicting_feature_group_count,
        "extra_rows": None,
        "rows_involved": conflicting_feature_rows,
    },
])
duplicate_summary


                                        category  ...  rows_involved
0  A_exact_duplicates_all_predictors_plus_target  ...          35086
1            B_identical_21_features_same_target  ...          35086
2      C_identical_21_features_different_targets  ...           6120

[3 rows x 4 columns]

## 7. Phân bố target

Class imbalance rất mạnh: class 0 chiếm đa số và class 1, Prediabetes, rất nhỏ. Vì vậy raw accuracy không đủ để làm metric chọn mô hình chính.


## 8. Phân bố feature

Range của feature được kiểm tra trước khi quyết định scaling có hữu ích hay không. Các range số khác nhau có thể ảnh hưởng đến gradient-based optimization, nhưng scaler chỉ được fit trên training split nếu được chọn.


In [6]:
feature_ranges = df[feature_columns].agg(["min", "max", "mean", "std", "nunique"]).T.reset_index().rename(columns={"index": "feature"})
feature_ranges["nunique"] = feature_ranges["nunique"].astype(int)

fig, axes = plt.subplots(5, 5, figsize=(14, 12))
axes = axes.ravel()
for axis, feature in zip(axes, feature_columns):
    values = df[feature]
    bins = min(30, int(values.nunique()))
    axis.hist(values, bins=bins)
    axis.set_title(feature, fontsize=8)
    axis.tick_params(labelsize=7)
for axis in axes[len(feature_columns):]:
    axis.axis("off")
fig.tight_layout()
dist_path = FIG_DIR / "feature_distributions.png"
fig.savefig(dist_path, dpi=150)
plt.close(fig)
print({"feature_distribution_figure": dist_path.relative_to(PROJECT_ROOT).as_posix()})
feature_ranges


{'feature_distribution_figure': 'results/figures/diabetes/hyperparameters/feature_distributions.png'}


                 feature   min   max       mean       std  nunique
0                 HighBP   0.0   1.0   0.429001  0.494934        2
1               HighChol   0.0   1.0   0.424121  0.494210        2
2              CholCheck   0.0   1.0   0.962670  0.189571        2
3                    BMI  12.0  98.0  28.382364  6.608694       84
4                 Smoker   0.0   1.0   0.443169  0.496761        2
5                 Stroke   0.0   1.0   0.040571  0.197294        2
6   HeartDiseaseorAttack   0.0   1.0   0.094186  0.292087        2
7           PhysActivity   0.0   1.0   0.756544  0.429169        2
8                 Fruits   0.0   1.0   0.634256  0.481639        2
9                Veggies   0.0   1.0   0.811420  0.391175        2
10     HvyAlcoholConsump   0.0   1.0   0.056197  0.230302        2
11         AnyHealthcare   0.0   1.0   0.951053  0.215759        2
12           NoDocbcCost   0.0   1.0   0.084177  0.277654        2
13               GenHlth   1.0   5.0   2.511392  1.068477     

## 9. Chia train / validation / test

Split stratified 70/15/15 deterministic được tạo với seed 42. Các file split đã lưu chứa row index và target label thay vì duplicate raw rows.


In [7]:
train_indices, temp_indices = train_test_split(
    df["row_index"],
    test_size=0.30,
    random_state=SEED,
    stratify=df[TARGET],
)
temp_targets = df.loc[temp_indices, TARGET]
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_targets,
)

split_frames = {
    "train": df.loc[train_indices, ["row_index", TARGET]].sort_values("row_index").reset_index(drop=True),
    "validation": df.loc[val_indices, ["row_index", TARGET]].sort_values("row_index").reset_index(drop=True),
    "test": df.loc[test_indices, ["row_index", TARGET]].sort_values("row_index").reset_index(drop=True),
}
split_frames["train"].to_csv(SPLIT_DIR / "diabetes_train.csv", index=False)
split_frames["validation"].to_csv(SPLIT_DIR / "diabetes_val.csv", index=False)
split_frames["test"].to_csv(SPLIT_DIR / "diabetes_test.csv", index=False)

sets = [set(frame["row_index"]) for frame in split_frames.values()]
overlap_count = len(sets[0] & sets[1]) + len(sets[0] & sets[2]) + len(sets[1] & sets[2])
accounted = sum(len(frame) for frame in split_frames.values())
assert accounted == len(df)
assert overlap_count == 0

split_distribution = []
for split_name, frame in split_frames.items():
    counts = frame[TARGET].value_counts().sort_index()
    for target, count in counts.items():
        split_distribution.append({
            "split": split_name,
            "target": int(target),
            "count": int(count),
            "proportion": float(count / len(frame)),
        })
split_distribution_df = pd.DataFrame(split_distribution)

print({"train": len(split_frames["train"]), "validation": len(split_frames["validation"]), "test": len(split_frames["test"]), "overlap_count": overlap_count})
split_distribution_df


{'train': 177576, 'validation': 38052, 'test': 38052, 'overlap_count': 0}


        split  target   count  proportion
0       train       0  149592    0.842411
1       train       1    3242    0.018257
2       train       2   24742    0.139332
3  validation       0   32056    0.842426
4  validation       1     694    0.018238
5  validation       2    5302    0.139336
6        test       0   32055    0.842400
7        test       1     695    0.018264
8        test       2    5302    0.139336

## 10. Preprocessing

Notebook so sánh không scaling với standard scaling được fit chỉ trên training split. Cùng scaler đã fit đó được áp dụng nguyên vẹn cho validation và test arrays, tránh leakage từ validation/test.


In [8]:
train_df = df.merge(split_frames["train"][["row_index"]], on="row_index", how="inner").sort_values("row_index").reset_index(drop=True)
val_df = df.merge(split_frames["validation"][["row_index"]], on="row_index", how="inner").sort_values("row_index").reset_index(drop=True)
test_df = df.merge(split_frames["test"][["row_index"]], on="row_index", how="inner").sort_values("row_index").reset_index(drop=True)

X_train_raw = train_df[feature_columns].to_numpy(dtype="float32")
X_val_raw = val_df[feature_columns].to_numpy(dtype="float32")
X_test_raw = test_df[feature_columns].to_numpy(dtype="float32")
y_train = train_df[TARGET].astype("int32").to_numpy()
y_val = val_df[TARGET].astype("int32").to_numpy()
y_test = test_df[TARGET].astype("int32").to_numpy()

scaler = StandardScaler()
X_train_standard = scaler.fit_transform(X_train_raw).astype("float32")
X_val_standard = scaler.transform(X_val_raw).astype("float32")
X_test_standard = scaler.transform(X_test_raw).astype("float32")

preprocessed_arrays = {
    "none": {"train": X_train_raw, "validation": X_val_raw, "test": X_test_raw},
    "standard": {"train": X_train_standard, "validation": X_val_standard, "test": X_test_standard},
}

print({
    "train_shape": X_train_raw.shape,
    "validation_shape": X_val_raw.shape,
    "test_shape": X_test_raw.shape,
    "scaler_fit_on": "train only",
})


{'train_shape': (177576, 21), 'validation_shape': (38052, 21), 'test_shape': (38052, 21), 'scaler_fit_on': 'train only'}


## 11. Biểu diễn Conv1D

Mỗi sample được biểu diễn là `(21, 1)`: 21 vị trí predictor và một channel. Shape progression của BasicCNN1D được hiển thị để xác minh sequence length không bị collapse.


In [9]:
def to_conv1d(array):
    return array.astype("float32")[..., np.newaxis]


sample_tensor_shape = to_conv1d(X_train_raw[:8]).shape
basic_shape_model = build_basic_cnn_1d(input_shape=(len(feature_columns), 1), num_classes=3)
shape_progression_df = pd.DataFrame(basic_shape_model.shape_progression)

print({"batch_tensor_shape_example": sample_tensor_shape, "model_parameters": int(basic_shape_model.count_params())})
shape_progression_df


{'batch_tensor_shape_example': (8, 21, 1), 'model_parameters': 2787}


                 stage  length  channels
0                input    21.0         1
1           conv1_same    21.0        16
2           pool1_same    11.0        16
3           conv2_same    11.0        32
4  global_average_pool     NaN        32

## 12. Phân tích class imbalance

Majority-class validation baseline cho thấy vì sao accuracy gây hiểu nhầm. Model selection nhấn mạnh macro F1, macro recall, per-class recall, đặc biệt cho Prediabetes và Diabetes.


In [10]:
def classification_summary(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )
    per_precision, per_recall, per_f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    return {
        "accuracy": float(np.mean(y_true == y_pred)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "class0_recall": float(per_recall[0]),
        "class1_prediabetes_recall": float(per_recall[1]),
        "class2_diabetes_recall": float(per_recall[2]),
    }


majority_class = int(pd.Series(y_train).mode().iloc[0])
majority_val_pred = np.full_like(y_val, majority_class)
majority_baseline = classification_summary(y_val, majority_val_pred)
majority_baseline["majority_class"] = majority_class
majority_confusion = confusion_matrix(y_val, majority_val_pred, labels=[0, 1, 2])
print(majority_baseline)
pd.DataFrame(majority_confusion, index=["true_0", "true_1", "true_2"], columns=["pred_0", "pred_1", "pred_2"])


{'accuracy': 0.8424261536844319, 'macro_precision': 0.2808087178948106, 'macro_recall': 0.3333333333333333, 'macro_f1': 0.30482493676423045, 'class0_recall': 1.0, 'class1_prediabetes_recall': 0.0, 'class2_diabetes_recall': 0.0, 'majority_class': 0}


        pred_0  pred_1  pred_2
true_0   32056       0       0
true_1     694       0       0
true_2    5302       0       0

## Experiment Runner

Tất cả thí nghiệm protocol bên dưới chỉ dùng BasicCNN1D. Official test split vẫn được giữ nguyên và không dùng cho tuning.


In [11]:
HP_PATH = HP_DIR / "diabetes_hyperparameters.csv"
all_results = []


def make_dataset(X, y, batch_size, shuffle=False, seed=SEED):
    dataset = tf.data.Dataset.from_tensor_slices((to_conv1d(X), y.astype("int32")))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(y), seed=seed, reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


def build_training_model(learning_rate, dropout_rate=0.0):
    if dropout_rate > 0:
        inputs = keras.Input(shape=(len(feature_columns), 1), name="tabular_sequence")
        x = keras.layers.Conv1D(16, 3, padding="same", activation="relu")(inputs)
        x = keras.layers.MaxPooling1D(pool_size=2, padding="same")(x)
        x = keras.layers.Conv1D(32, 3, padding="same", activation="relu")(x)
        x = keras.layers.GlobalAveragePooling1D()(x)
        x = keras.layers.Dense(32, activation="relu")(x)
        x = keras.layers.Dropout(dropout_rate, seed=SEED)(x)
        outputs = keras.layers.Dense(3, activation="softmax")(x)
        model = keras.Model(inputs, outputs, name=f"basic_cnn_1d_dropout_{dropout_rate}")
    else:
        model = build_basic_cnn_1d(input_shape=(len(feature_columns), 1), num_classes=3)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def evaluate_predictions(model, X, y, batch_size):
    eval_ds = make_dataset(X, y, batch_size=batch_size, shuffle=False)
    loss, accuracy = model.evaluate(eval_ds, verbose=0)
    probabilities = model.predict(eval_ds, verbose=0)
    y_pred = probabilities.argmax(axis=1)
    metrics = classification_summary(y, y_pred)
    metrics.update({"loss": float(loss), "accuracy": float(accuracy)})
    return metrics, y_pred


def run_experiment(
    experiment_type,
    candidate_value,
    preprocessing_key,
    learning_rate,
    batch_size,
    max_epochs,
    patience,
    class_weight_name="none",
    class_weight=None,
    dropout_rate=0.0,
):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    arrays = preprocessed_arrays[preprocessing_key]
    train_ds = make_dataset(arrays["train"], y_train, batch_size=batch_size, shuffle=True, seed=SEED)
    val_ds = make_dataset(arrays["validation"], y_val, batch_size=batch_size, shuffle=False)
    model = build_training_model(learning_rate=learning_rate, dropout_rate=dropout_rate)
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)]
    start = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds, epochs=max_epochs, callbacks=callbacks, verbose=0, class_weight=class_weight)
    training_time = time.perf_counter() - start
    train_metrics, _ = evaluate_predictions(model, arrays["train"], y_train, batch_size)
    val_metrics, y_val_pred = evaluate_predictions(model, arrays["validation"], y_val, batch_size)
    epochs_run = len(history.history["loss"])
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    result = {
        "experiment_type": experiment_type,
        "candidate_value": candidate_value,
        "preprocessing": preprocessing_key,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "max_epochs": max_epochs,
        "patience": patience,
        "epochs_run": epochs_run,
        "best_epoch": best_epoch,
        "class_weight": class_weight_name,
        "dropout_rate": dropout_rate,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_precision": val_metrics["macro_precision"],
        "val_macro_recall": val_metrics["macro_recall"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_class0_recall": val_metrics["class0_recall"],
        "val_class1_prediabetes_recall": val_metrics["class1_prediabetes_recall"],
        "val_class2_diabetes_recall": val_metrics["class2_diabetes_recall"],
        "training_time_seconds": float(training_time),
        "time_per_epoch_seconds": float(training_time / epochs_run),
        "seed": SEED,
    }
    all_results.append(result)
    pd.DataFrame(all_results).to_csv(HP_PATH, index=False)
    return result, history.history, y_val_pred


print({"experiment_runner": "ready", "results_path": HP_PATH.relative_to(PROJECT_ROOT).as_posix()})


{'experiment_runner': 'ready', 'results_path': 'results/hyperparameters/diabetes_hyperparameters.csv'}


## 13. Khảo sát learning rate

Trước khi tuning learning rate, preprocessing được chọn bằng cách so sánh không scaling với standard scaling fit trên train. Learning rate sau đó được chọn từ các candidate logarithmic bằng validation macro F1, macro recall, per-class recall, và validation loss.


In [12]:
scaling_candidates = ["none", "standard"]
scaling_histories = {}
for preprocessing_key in scaling_candidates:
    result, history, _ = run_experiment(
        experiment_type="scaling",
        candidate_value=preprocessing_key,
        preprocessing_key=preprocessing_key,
        learning_rate=1e-3,
        batch_size=512,
        max_epochs=5,
        patience=2,
    )
    scaling_histories[preprocessing_key] = history

scaling_df = pd.DataFrame([row for row in all_results if row["experiment_type"] == "scaling"])
selected_preprocessing = str(scaling_df.sort_values(["val_macro_f1", "val_macro_recall", "val_loss"], ascending=[False, False, True]).iloc[0]["preprocessing"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(scaling_df["preprocessing"], scaling_df["val_macro_f1"])
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Scaling comparison")
axes[1].bar(scaling_df["preprocessing"], scaling_df["val_class1_prediabetes_recall"])
axes[1].set_ylabel("class 1 recall")
axes[1].set_title("Prediabetes recall")
fig.tight_layout()
scaling_plot = FIG_DIR / "scaling_comparison.png"
fig.savefig(scaling_plot, dpi=150)
plt.close(fig)

print({"selected_preprocessing": selected_preprocessing, "plot": scaling_plot.relative_to(PROJECT_ROOT).as_posix()})
scaling_df[["candidate_value", "epochs_run", "val_loss", "val_accuracy", "val_macro_recall", "val_macro_f1", "val_class1_prediabetes_recall", "val_class2_diabetes_recall", "training_time_seconds"]]


{'selected_preprocessing': 'standard', 'plot': 'results/figures/diabetes/hyperparameters/scaling_comparison.png'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


  candidate_value  ...  training_time_seconds
0            none  ...               6.241314
1        standard  ...               5.767736

[2 rows x 9 columns]

In [13]:
lr_candidates = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
lr_histories = {}
for learning_rate in lr_candidates:
    result, history, _ = run_experiment(
        experiment_type="learning_rate",
        candidate_value=learning_rate,
        preprocessing_key=selected_preprocessing,
        learning_rate=learning_rate,
        batch_size=512,
        max_epochs=6,
        patience=2,
    )
    lr_histories[str(learning_rate)] = history

lr_df = pd.DataFrame([row for row in all_results if row["experiment_type"] == "learning_rate"])
best_lr_screen = float(lr_df.sort_values(["val_macro_f1", "val_macro_recall", "val_loss"], ascending=[False, False, True]).iloc[0]["learning_rate"])
extension_candidates = []
if best_lr_screen == min(lr_candidates):
    extension_candidates = [3e-5, 1e-5]
elif best_lr_screen == max(lr_candidates):
    extension_candidates = [3e-2]

for learning_rate in extension_candidates:
    result, history, _ = run_experiment(
        experiment_type="learning_rate",
        candidate_value=learning_rate,
        preprocessing_key=selected_preprocessing,
        learning_rate=learning_rate,
        batch_size=512,
        max_epochs=6,
        patience=2,
    )
    lr_histories[str(learning_rate)] = history

lr_df = pd.DataFrame([row for row in all_results if row["experiment_type"] == "learning_rate"])
selected_lr = float(lr_df.sort_values(["val_macro_f1", "val_macro_recall", "val_loss"], ascending=[False, False, True]).iloc[0]["learning_rate"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogx(lr_df["learning_rate"], lr_df["val_macro_f1"], marker="o")
axes[0].set_xlabel("learning rate")
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Learning rate vs macro F1")
axes[1].semilogx(lr_df["learning_rate"], lr_df["val_macro_recall"], marker="o")
axes[1].set_xlabel("learning rate")
axes[1].set_ylabel("validation macro recall")
axes[1].set_title("Learning rate vs macro recall")
fig.tight_layout()
lr_plot = FIG_DIR / "learning_rate_comparison.png"
fig.savefig(lr_plot, dpi=150)
plt.close(fig)

print({"selected_lr": selected_lr, "extension_candidates": extension_candidates, "plot": lr_plot.relative_to(PROJECT_ROOT).as_posix()})
lr_df.sort_values(["val_macro_f1", "val_macro_recall", "val_loss"], ascending=[False, False, True])[
    ["candidate_value", "epochs_run", "val_loss", "val_accuracy", "val_macro_recall", "val_macro_f1", "val_class1_prediabetes_recall", "val_class2_diabetes_recall", "training_time_seconds"]
]


{'selected_lr': 0.01, 'extension_candidates': [0.03], 'plot': 'results/figures/diabetes/hyperparameters/learning_rate_comparison.png'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

   candidate_value  ...  training_time_seconds
4           0.0100  ...               7.561975
3           0.0030  ...               7.703179
2           0.0010  ...               7.630449
5           0.0300  ...               7.323901
1           0.0003  ...               7.782706
0           0.0001  ...               7.794131

[6 rows x 9 columns]

## 14. Khảo sát batch size

Batch size được so sánh sau khi preprocessing và learning rate đã cố định. Quyết định dùng validation macro F1, macro recall, validation loss, và runtime như tie-breaker cuối.


In [14]:
batch_candidates = [256, 512, 1024, 2048]
batch_histories = {}
for batch_size in batch_candidates:
    result, history, _ = run_experiment(
        experiment_type="batch_size",
        candidate_value=batch_size,
        preprocessing_key=selected_preprocessing,
        learning_rate=selected_lr,
        batch_size=batch_size,
        max_epochs=5,
        patience=2,
    )
    batch_histories[str(batch_size)] = history

batch_df = pd.DataFrame([row for row in all_results if row["experiment_type"] == "batch_size"])
selected_batch = int(batch_df.sort_values(["val_macro_f1", "val_macro_recall", "val_loss", "time_per_epoch_seconds"], ascending=[False, False, True, True]).iloc[0]["batch_size"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(batch_df["batch_size"], batch_df["val_macro_f1"], marker="o")
axes[0].set_xlabel("batch size")
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Batch size vs macro F1")
axes[1].plot(batch_df["batch_size"], batch_df["time_per_epoch_seconds"], marker="o")
axes[1].set_xlabel("batch size")
axes[1].set_ylabel("seconds per epoch")
axes[1].set_title("Batch size runtime")
fig.tight_layout()
batch_plot = FIG_DIR / "batch_size_comparison.png"
fig.savefig(batch_plot, dpi=150)
plt.close(fig)

print({"selected_batch": selected_batch, "plot": batch_plot.relative_to(PROJECT_ROOT).as_posix()})
batch_df[["candidate_value", "epochs_run", "val_loss", "val_accuracy", "val_macro_recall", "val_macro_f1", "val_class1_prediabetes_recall", "val_class2_diabetes_recall", "time_per_epoch_seconds", "training_time_seconds"]]


{'selected_batch': 512, 'plot': 'results/figures/diabetes/hyperparameters/batch_size_comparison.png'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

   candidate_value  epochs_run  ...  time_per_epoch_seconds  training_time_seconds
0              256           5  ...                1.894035               9.470176
1              512           5  ...                1.329226               6.646129
2             1024           5  ...                1.082671               5.413357
3             2048           5  ...                0.852036               4.260178

[4 rows x 10 columns]

## 15. Khảo sát class weight

Class imbalance rất mạnh, nên class weighting được so sánh rõ ràng. Balanced inverse-frequency weights được tính chỉ từ training split. Selection ưu tiên macro/per-class recall thay vì raw accuracy.


In [15]:
classes = np.array([0, 1, 2])
balanced_values = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
balanced_weights = {int(label): float(weight) for label, weight in zip(classes, balanced_values)}
sqrt_values = np.sqrt(balanced_values)
sqrt_values = sqrt_values / np.mean(sqrt_values)
sqrt_balanced_weights = {int(label): float(weight) for label, weight in zip(classes, sqrt_values)}

class_weight_results = []
no_weight_reference = (
    pd.DataFrame(all_results)
    .query("experiment_type == 'batch_size' and batch_size == @selected_batch")
    .sort_values(["val_macro_f1", "val_macro_recall", "val_loss"], ascending=[False, False, True])
    .iloc[0]
    .to_dict()
)
no_weight_class_row = no_weight_reference.copy()
no_weight_class_row["experiment_type"] = "class_weight"
no_weight_class_row["candidate_value"] = "none"
no_weight_class_row["class_weight"] = "none"
all_results.append(no_weight_class_row)
pd.DataFrame(all_results).to_csv(HP_PATH, index=False)
class_weight_results.append(no_weight_class_row)

balanced_result, balanced_history, _ = run_experiment(
    experiment_type="class_weight",
    candidate_value="balanced_inverse_frequency",
    preprocessing_key=selected_preprocessing,
    learning_rate=selected_lr,
    batch_size=selected_batch,
    max_epochs=5,
    patience=2,
    class_weight_name="balanced_inverse_frequency",
    class_weight=balanced_weights,
)
class_weight_results.append(balanced_result)

balanced_overcorrected = (
    balanced_result["val_class1_prediabetes_recall"] > no_weight_reference["val_class1_prediabetes_recall"] + 0.20
    and balanced_result["val_class0_recall"] < no_weight_reference["val_class0_recall"] - 0.30
)
if balanced_overcorrected:
    sqrt_result, sqrt_history, _ = run_experiment(
        experiment_type="class_weight",
        candidate_value="sqrt_balanced_inverse_frequency",
        preprocessing_key=selected_preprocessing,
        learning_rate=selected_lr,
        batch_size=selected_batch,
        max_epochs=5,
        patience=2,
        class_weight_name="sqrt_balanced_inverse_frequency",
        class_weight=sqrt_balanced_weights,
    )
    class_weight_results.append(sqrt_result)

class_weight_df = pd.DataFrame(class_weight_results)
selected_class_weight_row = class_weight_df.sort_values(
    ["val_macro_f1", "val_macro_recall", "val_class1_prediabetes_recall", "val_class2_diabetes_recall", "val_loss"],
    ascending=[False, False, False, False, True],
).iloc[0]
selected_class_weight_name = str(selected_class_weight_row["candidate_value"])
selected_class_weight = None
if selected_class_weight_name == "balanced_inverse_frequency":
    selected_class_weight = balanced_weights
elif selected_class_weight_name == "sqrt_balanced_inverse_frequency":
    selected_class_weight = sqrt_balanced_weights

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(class_weight_df["candidate_value"].astype(str), class_weight_df["val_macro_f1"])
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Class-weight macro F1")
axes[0].tick_params(axis="x", rotation=25)
axes[1].plot(class_weight_df["candidate_value"].astype(str), class_weight_df["val_class0_recall"], marker="o", label="class 0")
axes[1].plot(class_weight_df["candidate_value"].astype(str), class_weight_df["val_class1_prediabetes_recall"], marker="o", label="class 1")
axes[1].plot(class_weight_df["candidate_value"].astype(str), class_weight_df["val_class2_diabetes_recall"], marker="o", label="class 2")
axes[1].set_ylabel("validation recall")
axes[1].set_title("Per-class recall")
axes[1].tick_params(axis="x", rotation=25)
axes[1].legend()
fig.tight_layout()
cw_plot = FIG_DIR / "class_weight_comparison.png"
fig.savefig(cw_plot, dpi=150)
plt.close(fig)

print({
    "balanced_weights": balanced_weights,
    "sqrt_balanced_weights": sqrt_balanced_weights,
    "balanced_overcorrected": bool(balanced_overcorrected),
    "selected_class_weight": selected_class_weight_name,
    "plot": cw_plot.relative_to(PROJECT_ROOT).as_posix(),
})
class_weight_df[["candidate_value", "epochs_run", "val_loss", "val_accuracy", "val_macro_recall", "val_macro_f1", "val_class0_recall", "val_class1_prediabetes_recall", "val_class2_diabetes_recall", "training_time_seconds"]]


{'balanced_weights': {0: 0.3956896090700037, 1: 18.257865515114126, 2: 2.3923692506668823}, 'sqrt_balanced_weights': {0: 0.2926355652959181, 1: 1.9878096096372029, 2: 0.7195548250668791}, 'balanced_overcorrected': False, 'selected_class_weight': 'balanced_inverse_frequency', 'plot': 'results/figures/diabetes/hyperparameters/class_weight_comparison.png'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


              candidate_value  ...  training_time_seconds
0                        none  ...               6.646129
1  balanced_inverse_frequency  ...               6.812412

[2 rows x 10 columns]

## 16. Regularization nếu cần

Dropout chỉ được thử nếu candidate đã chọn có khoảng cách train-vs-validation có ý nghĩa. Nó không được thêm mặc định.


In [16]:
selected_reference_pool = pd.DataFrame(all_results)
if selected_class_weight_name == "none":
    selected_reference = selected_class_weight_row
else:
    selected_reference = selected_reference_pool[
        (selected_reference_pool["experiment_type"] == "class_weight")
        & (selected_reference_pool["candidate_value"] == selected_class_weight_name)
    ].iloc[0]

accuracy_gap = float(selected_reference["train_accuracy"] - selected_reference["val_accuracy"])
macro_f1_gap = float(selected_reference["train_macro_f1"] - selected_reference["val_macro_f1"])
regularization_note = "No dropout experiment: training-vs-validation gap did not exceed the preset inspection threshold."
dropout_candidates_run = []
selected_dropout = 0.0

if accuracy_gap > 0.10 or macro_f1_gap > 0.10:
    regularization_note = "Training-vs-validation gap exceeded threshold; dropout candidates were compared."
    class_weight_arg = selected_class_weight
    class_weight_label = "none" if selected_class_weight_name == "none" else selected_class_weight_name
    for dropout_rate in [0.2, 0.4]:
        result, history, _ = run_experiment(
            experiment_type="dropout",
            candidate_value=dropout_rate,
            preprocessing_key=selected_preprocessing,
            learning_rate=selected_lr,
            batch_size=selected_batch,
            max_epochs=5,
            patience=2,
            class_weight_name=class_weight_label,
            class_weight=class_weight_arg,
            dropout_rate=dropout_rate,
        )
        dropout_candidates_run.append(result)
    dropout_df = pd.DataFrame(dropout_candidates_run)
    best_dropout = dropout_df.sort_values(["val_macro_f1", "val_macro_recall", "val_loss"], ascending=[False, False, True]).iloc[0]
    if best_dropout["val_macro_f1"] > selected_reference["val_macro_f1"]:
        selected_dropout = float(best_dropout["dropout_rate"])
        selected_reference = best_dropout
else:
    dropout_df = pd.DataFrame()

print({
    "accuracy_gap": accuracy_gap,
    "macro_f1_gap": macro_f1_gap,
    "regularization_note": regularization_note,
    "dropout_candidates_run": len(dropout_candidates_run),
    "selected_dropout": selected_dropout,
})
dropout_df if not dropout_df.empty else pd.DataFrame([{"dropout_candidates_run": 0, "reason": regularization_note}])


{'accuracy_gap': 0.0014360547065734863, 'macro_f1_gap': 0.0018562932366843476, 'regularization_note': 'No dropout experiment: training-vs-validation gap did not exceed the preset inspection threshold.', 'dropout_candidates_run': 0, 'selected_dropout': 0.0}


   dropout_candidates_run                                             reason
0                       0  No dropout experiment: training-vs-validation ...

## 17. Giao thức huấn luyện đã chọn

Bảng dưới đây đóng băng giao thức huấn luyện cấp dataset cho diabetes để dùng ở phần so sánh bốn kiến trúc. Giao thức được chọn mà không dùng test split.


In [17]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(HP_PATH, index=False)

selected_regularization = "none" if selected_dropout == 0 else f"dropout_{selected_dropout}"
selected_class_weight_value = "none" if selected_class_weight_name == "none" else selected_class_weight_name

protocol_rows = [
    {
        "decision": "feature order",
        "selected_value": "original CSV predictor column order",
        "classification": "data-determined",
        "evidence_or_reason": "Primary experiment preserves documented dataset feature order; no arbitrary reordering is used.",
    },
    {
        "decision": "Conv1D input representation",
        "selected_value": "(21, 1)",
        "classification": "architecture-determined",
        "evidence_or_reason": "BasicCNN1D expects a short sequence with one channel per predictor value.",
    },
    {
        "decision": "train/validation/test split",
        "selected_value": "70/15/15 stratified, seed 42",
        "classification": "operational bound",
        "evidence_or_reason": "Creates isolated validation and test roles while preserving class proportions.",
    },
    {
        "decision": "preprocessing",
        "selected_value": selected_preprocessing,
        "classification": "experimentally selected",
        "evidence_or_reason": "No scaling and train-fitted standard scaling were compared with BasicCNN1D on validation data.",
    },
    {
        "decision": "optimizer/protocol choice",
        "selected_value": "Adam optimizer family; BasicCNN1D-only protocol tuning; test untouched",
        "classification": "controlled protocol choice with theoretical rationale",
        "evidence_or_reason": "One optimizer family is held fixed while scalar protocol choices are selected by validation metrics.",
    },
    {
        "decision": "learning rate",
        "selected_value": selected_lr,
        "classification": "experimentally selected",
        "evidence_or_reason": "Selected from logarithmic candidates by validation macro F1/recall/loss; boundary extension applied if needed.",
    },
    {
        "decision": "batch size",
        "selected_value": selected_batch,
        "classification": "experimentally selected",
        "evidence_or_reason": "Selected by validation macro F1/recall/loss, with runtime as final tie-breaker.",
    },
    {
        "decision": "class weighting",
        "selected_value": selected_class_weight_value,
        "classification": "experimentally selected",
        "evidence_or_reason": "No weighting and train-only inverse-frequency weighting were compared using macro and per-class recall metrics.",
    },
    {
        "decision": "regularization",
        "selected_value": selected_regularization,
        "classification": "experimentally selected",
        "evidence_or_reason": regularization_note,
    },
    {
        "decision": "EarlyStopping",
        "selected_value": f"monitor val_loss, patience {int(selected_reference['patience'])}, restore_best_weights",
        "classification": "operational bound",
        "evidence_or_reason": "Limits CPU time and selects weights by validation loss only.",
    },
    {
        "decision": "max_epochs",
        "selected_value": int(selected_reference["max_epochs"]),
        "classification": "operational bound",
        "evidence_or_reason": "Upper bound for protocol-selection experiments; not claimed optimal.",
    },
]
protocol_df = pd.DataFrame(protocol_rows)
protocol_path = HP_DIR / "diabetes_selected_protocol.csv"
protocol_df.to_csv(protocol_path, index=False)

runtime_path = HP_DIR / "diabetes_protocol_runtime.json"
runtime_path.write_text(json.dumps({
    "python_executable": actual_python,
    "tensorflow_version": tf.__version__,
    "experiment_rows": len(results_df),
    "recorded_training_seconds": float(results_df["training_time_seconds"].sum()),
    "selected_preprocessing": selected_preprocessing,
    "selected_learning_rate": selected_lr,
    "selected_batch_size": selected_batch,
    "selected_class_weight": selected_class_weight_value,
    "selected_regularization": selected_regularization,
}, indent=2), encoding="utf-8")

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
selected_arrays = preprocessed_arrays[selected_preprocessing]
selected_train_ds = make_dataset(selected_arrays["train"], y_train, batch_size=selected_batch, shuffle=True, seed=SEED)
selected_val_ds = make_dataset(selected_arrays["validation"], y_val, batch_size=selected_batch, shuffle=False)
selected_model = build_training_model(learning_rate=selected_lr, dropout_rate=selected_dropout)
selected_callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=int(selected_reference["patience"]), restore_best_weights=True)]
selected_model.fit(
    selected_train_ds,
    validation_data=selected_val_ds,
    epochs=int(selected_reference["max_epochs"]),
    callbacks=selected_callbacks,
    verbose=0,
    class_weight=selected_class_weight,
)
selected_probabilities = selected_model.predict(selected_val_ds, verbose=0)
selected_val_pred = selected_probabilities.argmax(axis=1)
selected_confusion = confusion_matrix(y_val, selected_val_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(selected_confusion, cmap="Blues")
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(["0 No diabetes", "1 Prediabetes", "2 Diabetes"], rotation=30, ha="right")
ax.set_yticklabels(["0 No diabetes", "1 Prediabetes", "2 Diabetes"])
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Selected protocol validation confusion matrix")
for row in range(3):
    for col in range(3):
        ax.text(col, row, int(selected_confusion[row, col]), ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
selected_cm_path = FIG_DIR / "selected_protocol_validation_confusion_matrix.png"
fig.savefig(selected_cm_path, dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
final_comparison = results_df[results_df["experiment_type"].isin(["scaling", "learning_rate", "batch_size", "class_weight", "dropout"])]
axes[0].scatter(final_comparison["val_macro_recall"], final_comparison["val_macro_f1"], alpha=0.8)
axes[0].set_xlabel("validation macro recall")
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Protocol candidates")
axes[1].bar(["majority", "selected"], [majority_baseline["macro_f1"], float(selected_reference["val_macro_f1"])])
axes[1].set_ylabel("validation macro F1")
axes[1].set_title("Majority baseline vs selected")
fig.tight_layout()
summary_plot = FIG_DIR / "selected_protocol_summary.png"
fig.savefig(summary_plot, dpi=150)
plt.close(fig)

print({
    "hyperparameter_results": HP_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "selected_protocol": protocol_path.relative_to(PROJECT_ROOT).as_posix(),
    "runtime_summary": runtime_path.relative_to(PROJECT_ROOT).as_posix(),
    "selected_validation_confusion_matrix": selected_cm_path.relative_to(PROJECT_ROOT).as_posix(),
    "summary_plot": summary_plot.relative_to(PROJECT_ROOT).as_posix(),
    "experiment_rows": len(results_df),
})
protocol_df


{'hyperparameter_results': 'results/hyperparameters/diabetes_hyperparameters.csv', 'selected_protocol': 'results/hyperparameters/diabetes_selected_protocol.csv', 'runtime_summary': 'results/hyperparameters/diabetes_protocol_runtime.json', 'selected_validation_confusion_matrix': 'results/figures/diabetes/hyperparameters/selected_protocol_validation_confusion_matrix.png', 'summary_plot': 'results/figures/diabetes/hyperparameters/selected_protocol_summary.png', 'experiment_rows': 14}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


                       decision  ...                                 evidence_or_reason
0                 feature order  ...  Primary experiment preserves documented datase...
1   Conv1D input representation  ...  BasicCNN1D expects a short sequence with one c...
2   train/validation/test split  ...  Creates isolated validation and test roles whi...
3                 preprocessing  ...  No scaling and train-fitted standard scaling w...
4     optimizer/protocol choice  ...  One optimizer family is held fixed while scala...
5                 learning rate  ...  Selected from logarithmic candidates by valida...
6                    batch size  ...  Selected by validation macro F1/recall/loss, w...
7               class weighting  ...  No weighting and train-only inverse-frequency ...
8                regularization  ...  No dropout experiment: training-vs-validation ...
9                 EarlyStopping  ...  Limits CPU time and selects weights by validat...
10                   max_epochs 

## 18. So sánh cuối cùng bốn mô hình Diabetes

Giao thức diabetes đã chọn được đóng băng trước phần so sánh kiến trúc cuối. Phần này huấn luyện BasicCNN1D, AlexNetInspired1D, VGGInspired1D, và ResNetInspired1D từ đầu với cùng row-index split, thứ tự feature gốc, standard scaling fit trên train, class weights cân bằng tính từ train, optimizer family, learning rate, batch size, EarlyStopping, seed policy, và định nghĩa metric. Test split chỉ được đánh giá sau khi tất cả checkpoint đã được chọn bằng validation.


In [18]:
from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from models.architectures import (
    build_alexnet_inspired_1d,
    build_basic_cnn_1d,
    build_resnet_inspired_1d,
    build_vgg_inspired_1d,
)

SEED = 42
EXPECTED_PYTHON = "C:/Users/anhca/anaconda3/envs/tf312/python.exe"
actual_python = sys.executable.replace("\\", "/")
assert actual_python.lower() == EXPECTED_PYTHON.lower(), actual_python

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "AGENTS.md").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("A05 project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "datasets" / "diabetes" / "diabetes_012_health_indicators_BRFSS2015.csv"
SPLIT_DIR = PROJECT_ROOT / "results" / "splits"
HP_DIR = PROJECT_ROOT / "results" / "hyperparameters"
METRIC_DIR = PROJECT_ROOT / "results" / "metrics"
FIG_ROOT = PROJECT_ROOT / "results" / "figures" / "diabetes"
CURVE_DIR = FIG_ROOT / "training_curves"
CM_DIR = FIG_ROOT / "confusion_matrices"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"
for path in [METRIC_DIR, CURVE_DIR, CM_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.insert(0, "row_index", np.arange(len(df), dtype=np.int64))
TARGET = "Diabetes_012"
feature_columns = [column for column in df.columns if column not in ["row_index", TARGET]]
protocol_df = pd.read_csv(HP_DIR / "diabetes_selected_protocol.csv")
protocol = dict(zip(protocol_df["decision"], protocol_df["selected_value"]))

assert protocol["feature order"] == "original CSV predictor column order"
assert protocol["Conv1D input representation"] == "(21, 1)"
assert protocol["preprocessing"] == "standard"
assert protocol["class weighting"] == "balanced_inverse_frequency"

selected_lr = float(protocol["learning rate"])
selected_batch_size = int(protocol["batch size"])
selected_patience = int(str(protocol["EarlyStopping"]).split("patience ")[1].split(",")[0])
selected_max_epochs = int(protocol["max_epochs"])

train_ids = pd.read_csv(SPLIT_DIR / "diabetes_train.csv")
val_ids = pd.read_csv(SPLIT_DIR / "diabetes_val.csv")
test_ids = pd.read_csv(SPLIT_DIR / "diabetes_test.csv")
train_df = df.merge(train_ids[["row_index"]], on="row_index", how="inner").sort_values("row_index").reset_index(drop=True)
val_df = df.merge(val_ids[["row_index"]], on="row_index", how="inner").sort_values("row_index").reset_index(drop=True)
test_df = df.merge(test_ids[["row_index"]], on="row_index", how="inner").sort_values("row_index").reset_index(drop=True)

sets = [set(frame["row_index"]) for frame in [train_df, val_df, test_df]]
overlap_count = len(sets[0] & sets[1]) + len(sets[0] & sets[2]) + len(sets[1] & sets[2])
assert len(train_df) == 177576 and len(val_df) == 38052 and len(test_df) == 38052
assert overlap_count == 0
assert len(feature_columns) == 21

X_train_raw = train_df[feature_columns].to_numpy(dtype="float32")
X_val_raw = val_df[feature_columns].to_numpy(dtype="float32")
X_test_raw = test_df[feature_columns].to_numpy(dtype="float32")
y_train = train_df[TARGET].astype("int32").to_numpy()
y_val = val_df[TARGET].astype("int32").to_numpy()
y_test = test_df[TARGET].astype("int32").to_numpy()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype("float32")
X_val = scaler.transform(X_val_raw).astype("float32")
X_test = scaler.transform(X_test_raw).astype("float32")

class_labels = np.array([0, 1, 2])
class_names = ["No diabetes", "Prediabetes", "Diabetes"]
balanced_values = compute_class_weight(class_weight="balanced", classes=class_labels, y=y_train)
selected_class_weight = {int(label): float(weight) for label, weight in zip(class_labels, balanced_values)}

print({
    "python_executable": actual_python,
    "tensorflow_version": tf.__version__,
    "devices": [str(device) for device in tf.config.list_physical_devices()],
    "train": len(train_df),
    "validation": len(val_df),
    "test": len(test_df),
    "features": len(feature_columns),
    "learning_rate": selected_lr,
    "batch_size": selected_batch_size,
    "max_epochs": selected_max_epochs,
    "patience": selected_patience,
    "class_weight": selected_class_weight,
})


{'python_executable': 'C:/Users/anhca/anaconda3/envs/tf312/python.exe', 'tensorflow_version': '2.21.0', 'devices': ["PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')"], 'train': 177576, 'validation': 38052, 'test': 38052, 'features': 21, 'learning_rate': 0.01, 'batch_size': 512, 'max_epochs': 5, 'patience': 2, 'class_weight': {0: 0.3956896090700037, 1: 18.257865515114126, 2: 2.3923692506668823}}


## 19. Pipeline tabular Conv1D đã đóng băng

So sánh cuối giữ nguyên thứ tự 21 feature từ CSV. Standard scaling được fit trên training split, sau đó áp dụng nguyên vẹn cho validation và test. Mỗi sample được biểu diễn là `(21, 1)`.


In [19]:
def to_conv1d(array):
    return array.astype("float32")[..., np.newaxis]


def make_dataset(X, y, batch_size, shuffle=False, seed=SEED):
    dataset = tf.data.Dataset.from_tensor_slices((to_conv1d(X), y.astype("int32")))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(y), seed=seed, reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_eval_ds = make_dataset(X_train, y_train, selected_batch_size, shuffle=False)
val_ds = make_dataset(X_val, y_val, selected_batch_size, shuffle=False)
test_ds = make_dataset(X_test, y_test, selected_batch_size, shuffle=False)

print({
    "conv1d_train_shape": to_conv1d(X_train[:8]).shape,
    "feature_order_first_five": feature_columns[:5],
    "scaler_fit_on": "train only",
})


{'conv1d_train_shape': (8, 21, 1), 'feature_order_first_five': ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker'], 'scaler_fit_on': 'train only'}


## 20. Huấn luyện bốn family kiến trúc Conv1D

Mỗi kiến trúc được huấn luyện một lần dưới protocol đã đóng băng. Class weights là balanced inverse-frequency weights tính từ training split. Validation loss chọn checkpoint thông qua EarlyStopping và ModelCheckpoint.


In [20]:
MODEL_SPECS = [
    {
        "family": "BasicCNN1D",
        "key": "basic",
        "builder": build_basic_cnn_1d,
        "checkpoint": CHECKPOINT_DIR / "diabetes_basic.keras",
    },
    {
        "family": "AlexNetInspired1D",
        "key": "alexnet_inspired",
        "builder": build_alexnet_inspired_1d,
        "checkpoint": CHECKPOINT_DIR / "diabetes_alexnet_inspired.keras",
    },
    {
        "family": "VGGInspired1D",
        "key": "vgg_inspired",
        "builder": build_vgg_inspired_1d,
        "checkpoint": CHECKPOINT_DIR / "diabetes_vgg_inspired.keras",
    },
    {
        "family": "ResNetInspired1D",
        "key": "resnet_inspired",
        "builder": build_resnet_inspired_1d,
        "checkpoint": CHECKPOINT_DIR / "diabetes_resnet_inspired.keras",
    },
]


def compile_model(builder):
    model = builder(input_shape=(len(feature_columns), 1), num_classes=3)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=selected_lr),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def plot_history(history_df, family, key):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history_df["epoch"], history_df["loss"], marker="o", label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].set_title(f"{family}: loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["accuracy"], marker="o", label="train")
    axes[1].plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="validation")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")
    axes[1].set_title(f"{family}: accuracy")
    axes[1].legend()
    fig.tight_layout()
    path = CURVE_DIR / f"{key}_history.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


training_rows = []
comparison_start = time.perf_counter()

for index, spec in enumerate(MODEL_SPECS):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    train_model_ds = make_dataset(X_train, y_train, selected_batch_size, shuffle=True, seed=SEED)
    model = compile_model(spec["builder"])
    parameter_count = int(model.count_params())
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=selected_patience,
            restore_best_weights=True,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=spec["checkpoint"],
            monitor="val_loss",
            save_best_only=True,
        ),
    ]
    start = time.perf_counter()
    history = model.fit(
        train_model_ds,
        validation_data=val_ds,
        epochs=selected_max_epochs,
        callbacks=callbacks,
        verbose=0,
        class_weight=selected_class_weight,
    )
    training_time = time.perf_counter() - start
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    history_path = METRIC_DIR / f"diabetes_{spec['key']}_history.csv"
    history_df.to_csv(history_path, index=False)
    curve_path = plot_history(history_df, spec["family"], spec["key"])
    best_index = int(history_df["val_loss"].idxmin())
    best_epoch = int(history_df.loc[best_index, "epoch"])
    early_stopping_activated = len(history_df) < selected_max_epochs
    training_rows.append({
        "model_family": spec["family"],
        "key": spec["key"],
        "checkpoint_path": spec["checkpoint"].relative_to(PROJECT_ROOT).as_posix(),
        "parameter_count": parameter_count,
        "epochs_run": int(len(history_df)),
        "best_epoch": best_epoch,
        "early_stopping_activated": bool(early_stopping_activated),
        "train_loss_best_epoch_history": float(history_df.loc[best_index, "loss"]),
        "train_loss_final_epoch_history": float(history_df.iloc[-1]["loss"]),
        "train_accuracy_best_epoch_history": float(history_df.loc[best_index, "accuracy"]),
        "train_accuracy_final_epoch_history": float(history_df.iloc[-1]["accuracy"]),
        "validation_loss_best_epoch_history": float(history_df.loc[best_index, "val_loss"]),
        "validation_accuracy_best_epoch_history": float(history_df.loc[best_index, "val_accuracy"]),
        "training_time_seconds": float(training_time),
        "history_path": history_path.relative_to(PROJECT_ROOT).as_posix(),
        "history_figure": curve_path.relative_to(PROJECT_ROOT).as_posix(),
        "learning_rate": selected_lr,
        "batch_size": selected_batch_size,
        "max_epochs": selected_max_epochs,
        "patience": selected_patience,
        "seed": SEED,
    })
    pd.DataFrame(training_rows).to_csv(METRIC_DIR / "diabetes_training_summaries.csv", index=False)
    print({
        "trained": spec["family"],
        "parameters": parameter_count,
        "epochs_run": int(len(history_df)),
        "best_epoch": best_epoch,
        "early_stopping_activated": bool(early_stopping_activated),
        "training_time_seconds": round(training_time, 2),
        "completed": f"{index + 1}/{len(MODEL_SPECS)}",
    })

training_summary_df = pd.DataFrame(training_rows)
training_summary_df[[
    "model_family",
    "parameter_count",
    "epochs_run",
    "best_epoch",
    "early_stopping_activated",
    "training_time_seconds",
]]


{'trained': 'BasicCNN1D', 'parameters': 2787, 'epochs_run': 5, 'best_epoch': 5, 'early_stopping_activated': False, 'training_time_seconds': 6.47, 'completed': '1/4'}
{'trained': 'AlexNetInspired1D', 'parameters': 29635, 'epochs_run': 5, 'best_epoch': 3, 'early_stopping_activated': False, 'training_time_seconds': 12.36, 'completed': '2/4'}
{'trained': 'VGGInspired1D', 'parameters': 38299, 'epochs_run': 3, 'best_epoch': 1, 'early_stopping_activated': True, 'training_time_seconds': 10.21, 'completed': '3/4'}
{'trained': 'ResNetInspired1D', 'parameters': 44387, 'epochs_run': 3, 'best_epoch': 1, 'early_stopping_activated': True, 'training_time_seconds': 11.51, 'completed': '4/4'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

        model_family  ...  training_time_seconds
0         BasicCNN1D  ...               6.467587
1  AlexNetInspired1D  ...              12.364384
2      VGGInspired1D  ...              10.209813
3   ResNetInspired1D  ...              11.513357

[4 rows x 6 columns]

## 21. Đánh giá official test

Sau khi cả bốn checkpoint được chọn, cell này đánh giá test split chưa bị chạm tới. Majority-class baseline được đưa vào để diễn giải raw accuracy trong bối cảnh class imbalance.


In [21]:
def split_metrics(model, dataset, y_true):
    start = time.perf_counter()
    probabilities = model.predict(dataset, verbose=0)
    inference_time = time.perf_counter() - start
    loss, accuracy = model.evaluate(dataset, verbose=0)
    y_pred = probabilities.argmax(axis=1)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="weighted",
        zero_division=0,
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    return {
        "loss": float(loss),
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "per_precision": per_precision,
        "per_recall": per_recall,
        "per_f1": per_f1,
        "per_support": per_support,
        "y_pred": y_pred,
        "probabilities": probabilities,
        "inference_time_seconds": float(inference_time),
    }


def baseline_metrics(y_true, majority_class):
    y_pred = np.full_like(y_true, majority_class)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="weighted",
        zero_division=0,
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    return {
        "test_accuracy": float(np.mean(y_true == y_pred)),
        "test_macro_precision": float(macro_precision),
        "test_macro_recall": float(macro_recall),
        "test_macro_f1": float(macro_f1),
        "test_weighted_f1": float(weighted_f1),
        "per_precision": per_precision,
        "per_recall": per_recall,
        "per_f1": per_f1,
        "per_support": per_support,
        "y_pred": y_pred,
    }


def plot_confusion(matrix, title, path):
    fig, ax = plt.subplots(figsize=(5.8, 4.8))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(["0 No diabetes", "1 Prediabetes", "2 Diabetes"], rotation=30, ha="right")
    ax.set_yticklabels(["0 No diabetes", "1 Prediabetes", "2 Diabetes"])
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)
    for row in range(3):
        for col in range(3):
            ax.text(col, row, int(matrix[row, col]), ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


metric_rows = []
per_class_rows = []

majority_class = int(pd.Series(y_train).mode().iloc[0])
baseline = baseline_metrics(y_test, majority_class)
baseline_matrix = confusion_matrix(y_test, baseline["y_pred"], labels=[0, 1, 2])
baseline_cm_path = plot_confusion(
    baseline_matrix,
    "Majority-class baseline: official test confusion matrix",
    CM_DIR / "majority_baseline_confusion_matrix.png",
)
metric_rows.append({
    "model_family": "MajorityClassBaseline",
    "key": "majority_baseline",
    "test_loss": np.nan,
    "test_accuracy": baseline["test_accuracy"],
    "test_macro_precision": baseline["test_macro_precision"],
    "test_macro_recall": baseline["test_macro_recall"],
    "test_macro_f1": baseline["test_macro_f1"],
    "test_weighted_f1": baseline["test_weighted_f1"],
    "test_class0_recall": float(baseline["per_recall"][0]),
    "test_class1_prediabetes_recall": float(baseline["per_recall"][1]),
    "test_class2_diabetes_recall": float(baseline["per_recall"][2]),
    "trainable_parameter_count": 0,
    "best_epoch": 0,
    "epochs_run": 0,
    "early_stopping_activated": False,
    "training_time_seconds": 0.0,
    "test_inference_time_seconds": 0.0,
    "test_inference_ms_per_sample": 0.0,
    "checkpoint_path": "",
    "confusion_matrix_figure": baseline_cm_path.relative_to(PROJECT_ROOT).as_posix(),
})
for class_index, class_name in enumerate(class_names):
    per_class_rows.append({
        "model_family": "MajorityClassBaseline",
        "key": "majority_baseline",
        "class_label": class_index,
        "class_name": class_name,
        "precision": float(baseline["per_precision"][class_index]),
        "recall": float(baseline["per_recall"][class_index]),
        "f1": float(baseline["per_f1"][class_index]),
        "support": int(baseline["per_support"][class_index]),
    })

for spec in MODEL_SPECS:
    tf.keras.backend.clear_session()
    model = keras.models.load_model(spec["checkpoint"])
    train_metrics = split_metrics(model, train_eval_ds, y_train)
    val_metrics = split_metrics(model, val_ds, y_val)
    test_metrics = split_metrics(model, test_ds, y_test)
    matrix = confusion_matrix(y_test, test_metrics["y_pred"], labels=[0, 1, 2])
    cm_path = plot_confusion(
        matrix,
        f"{spec['family']}: official test confusion matrix",
        CM_DIR / f"{spec['key']}_confusion_matrix.png",
    )
    prediction_df = test_df[["row_index", TARGET]].copy()
    prediction_df["predicted_label"] = test_metrics["y_pred"].astype(int)
    prediction_df["max_probability"] = test_metrics["probabilities"].max(axis=1)
    prediction_path = METRIC_DIR / f"diabetes_{spec['key']}_test_predictions.csv"
    prediction_df.to_csv(prediction_path, index=False)
    training_info = training_summary_df[training_summary_df["key"] == spec["key"]].iloc[0].to_dict()
    metric_rows.append({
        "model_family": spec["family"],
        "key": spec["key"],
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "validation_loss": val_metrics["loss"],
        "validation_accuracy": val_metrics["accuracy"],
        "test_loss": test_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_precision": test_metrics["macro_precision"],
        "test_macro_recall": test_metrics["macro_recall"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "test_class0_recall": float(test_metrics["per_recall"][0]),
        "test_class1_prediabetes_recall": float(test_metrics["per_recall"][1]),
        "test_class2_diabetes_recall": float(test_metrics["per_recall"][2]),
        "trainable_parameter_count": int(training_info["parameter_count"]),
        "best_epoch": int(training_info["best_epoch"]),
        "epochs_run": int(training_info["epochs_run"]),
        "early_stopping_activated": bool(training_info["early_stopping_activated"]),
        "training_time_seconds": float(training_info["training_time_seconds"]),
        "test_inference_time_seconds": test_metrics["inference_time_seconds"],
        "test_inference_ms_per_sample": test_metrics["inference_time_seconds"] / len(y_test) * 1000,
        "checkpoint_path": training_info["checkpoint_path"],
        "confusion_matrix_figure": cm_path.relative_to(PROJECT_ROOT).as_posix(),
        "prediction_path": prediction_path.relative_to(PROJECT_ROOT).as_posix(),
    })
    for class_index, class_name in enumerate(class_names):
        per_class_rows.append({
            "model_family": spec["family"],
            "key": spec["key"],
            "class_label": class_index,
            "class_name": class_name,
            "precision": float(test_metrics["per_precision"][class_index]),
            "recall": float(test_metrics["per_recall"][class_index]),
            "f1": float(test_metrics["per_f1"][class_index]),
            "support": int(test_metrics["per_support"][class_index]),
        })
    print({
        "evaluated": spec["family"],
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "test_macro_f1": round(test_metrics["macro_f1"], 4),
        "prediabetes_recall": round(float(test_metrics["per_recall"][1]), 4),
        "diabetes_recall": round(float(test_metrics["per_recall"][2]), 4),
    })

models_df = pd.DataFrame(metric_rows)
per_class_df = pd.DataFrame(per_class_rows)
models_path = METRIC_DIR / "diabetes_models.csv"
per_class_path = METRIC_DIR / "diabetes_per_class_metrics.csv"
models_df.to_csv(models_path, index=False)
per_class_df.to_csv(per_class_path, index=False)

total_comparison_runtime = time.perf_counter() - comparison_start
runtime_path = METRIC_DIR / "diabetes_final_runtime.json"
runtime_path.write_text(json.dumps({
    "total_runtime_seconds": total_comparison_runtime,
    "model_training_seconds": models_df[models_df["model_family"] != "MajorityClassBaseline"][["model_family", "training_time_seconds"]].to_dict(orient="records"),
    "model_test_inference_seconds": models_df[models_df["model_family"] != "MajorityClassBaseline"][["model_family", "test_inference_time_seconds"]].to_dict(orient="records"),
    "python_executable": actual_python,
    "tensorflow_version": tf.__version__,
}, indent=2), encoding="utf-8")

models_df[[
    "model_family",
    "test_loss",
    "test_accuracy",
    "test_macro_precision",
    "test_macro_recall",
    "test_macro_f1",
    "test_weighted_f1",
    "test_class0_recall",
    "test_class1_prediabetes_recall",
    "test_class2_diabetes_recall",
    "trainable_parameter_count",
    "best_epoch",
    "training_time_seconds",
    "test_inference_time_seconds",
]]


{'evaluated': 'BasicCNN1D', 'test_accuracy': 0.6953, 'test_macro_f1': 0.4392, 'prediabetes_recall': 0.2518, 'diabetes_recall': 0.5273}
{'evaluated': 'AlexNetInspired1D', 'test_accuracy': 0.608, 'test_macro_f1': 0.4075, 'prediabetes_recall': 0.5094, 'diabetes_recall': 0.3412}
{'evaluated': 'VGGInspired1D', 'test_accuracy': 0.6793, 'test_macro_f1': 0.4261, 'prediabetes_recall': 0.1065, 'diabetes_recall': 0.6777}
{'evaluated': 'ResNetInspired1D', 'test_accuracy': 0.7124, 'test_macro_f1': 0.4356, 'prediabetes_recall': 0.0921, 'diabetes_recall': 0.652}


            model_family  ...  test_inference_time_seconds
0  MajorityClassBaseline  ...                     0.000000
1             BasicCNN1D  ...                     0.092471
2      AlexNetInspired1D  ...                     0.146516
3          VGGInspired1D  ...                     0.191927
4       ResNetInspired1D  ...                     0.251095

[5 rows x 14 columns]

## 22. So sánh cuối và độ phức tạp

Accuracy không phải kết luận chính vì class 0 chiếm đa số dataset. Macro F1, macro recall, per-class recall, đặc biệt Prediabetes và Diabetes recall, được nhấn mạnh cùng với parameter count, training time, và inference time.


In [22]:
model_only_df = models_df[models_df["model_family"] != "MajorityClassBaseline"].copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(models_df["model_family"], models_df["test_macro_f1"])
axes[0].set_ylabel("official test macro F1")
axes[0].set_title("Macro F1 including majority baseline")
axes[0].tick_params(axis="x", rotation=35)
axes[1].bar(models_df["model_family"], models_df["test_class1_prediabetes_recall"].fillna(0.0))
axes[1].set_ylabel("Prediabetes recall")
axes[1].set_title("Class 1 recall")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
metric_plot = FIG_ROOT / "diabetes_final_metrics.png"
fig.savefig(metric_plot, dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(model_only_df["trainable_parameter_count"], model_only_df["test_macro_f1"], s=80)
for _, row in model_only_df.iterrows():
    axes[0].annotate(row["model_family"].replace("Inspired1D", ""), (row["trainable_parameter_count"], row["test_macro_f1"]), fontsize=8)
axes[0].set_xscale("log")
axes[0].set_xlabel("trainable parameters (log scale)")
axes[0].set_ylabel("test macro F1")
axes[0].set_title("Performance vs parameter count")
axes[1].scatter(model_only_df["training_time_seconds"], model_only_df["test_macro_f1"], s=80)
for _, row in model_only_df.iterrows():
    axes[1].annotate(row["model_family"].replace("Inspired1D", ""), (row["training_time_seconds"], row["test_macro_f1"]), fontsize=8)
axes[1].set_xlabel("training time (seconds)")
axes[1].set_ylabel("test macro F1")
axes[1].set_title("Performance vs training time")
fig.tight_layout()
complexity_plot = FIG_ROOT / "diabetes_complexity_tradeoffs.png"
fig.savefig(complexity_plot, dpi=150)
plt.close(fig)

display_table = models_df[[
    "model_family",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
    "test_class0_recall",
    "test_class1_prediabetes_recall",
    "test_class2_diabetes_recall",
    "trainable_parameter_count",
    "training_time_seconds",
    "test_inference_ms_per_sample",
    "best_epoch",
]].copy()
for column in [
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
    "test_class0_recall",
    "test_class1_prediabetes_recall",
    "test_class2_diabetes_recall",
    "test_inference_ms_per_sample",
]:
    display_table[column] = display_table[column].round(4)
display_table["training_time_seconds"] = display_table["training_time_seconds"].round(2)

print({
    "metric_plot": metric_plot.relative_to(PROJECT_ROOT).as_posix(),
    "complexity_plot": complexity_plot.relative_to(PROJECT_ROOT).as_posix(),
    "total_runtime_seconds": round(total_comparison_runtime, 2),
})
display_table


{'metric_plot': 'results/figures/diabetes/diabetes_final_metrics.png', 'complexity_plot': 'results/figures/diabetes/diabetes_complexity_tradeoffs.png', 'total_runtime_seconds': 55.66}


            model_family  ...  best_epoch
0  MajorityClassBaseline  ...           0
1             BasicCNN1D  ...           5
2      AlexNetInspired1D  ...           3
3          VGGInspired1D  ...           1
4       ResNetInspired1D  ...           1

[5 rows x 11 columns]

## 23. Metric theo từng lớp

Class 0 là No diabetes, class 1 là Prediabetes, và class 2 là Diabetes. Minority-class recall được diễn giải trực tiếp thay vì bị che bởi aggregate accuracy.


In [23]:
per_class_df.pivot_table(
    index=["model_family", "class_name"],
    values=["precision", "recall", "f1"],
    aggfunc="first",
).round(4)


                                       f1  precision  recall
model_family          class_name                            
AlexNetInspired1D     Diabetes     0.3915     0.4591  0.3412
                      No diabetes  0.7756     0.9522  0.6543
                      Prediabetes  0.0554     0.0293  0.5094
BasicCNN1D            Diabetes     0.4388     0.3758  0.5273
                      No diabetes  0.8233     0.9395  0.7327
                      Prediabetes  0.0555     0.0312  0.2518
MajorityClassBaseline Diabetes     0.0000     0.0000  0.0000
                      No diabetes  0.9145     0.8424  1.0000
                      Prediabetes  0.0000     0.0000  0.0000
ResNetInspired1D      Diabetes     0.4437     0.3363  0.6520
                      No diabetes  0.8234     0.9347  0.7358
                      Prediabetes  0.0396     0.0252  0.0921
VGGInspired1D         Diabetes     0.4466     0.3331  0.6777
                      No diabetes  0.7985     0.9438  0.6920
                      Pr

## 24. Giới hạn của Tabular Conv1D

Feature adjacency trong dataset tabular này là nhân tạo: các cột đứng cạnh nhau chỉ vì schema CSV, không phải vì chúng có cấu trúc không gian cục bộ tự nhiên. Hiệu năng Conv1D ở đây không chứng minh CNN là tối ưu cho phân loại diabetes dạng tabular. Thí nghiệm này đánh giá adaptation bắt buộc của các ý tưởng kiến trúc CNN trong một protocol có kiểm soát.


## 25. Diễn giải cuối dựa trên số đo

Majority-class baseline dự đoán mọi sample test là class 0, No diabetes. Nó đạt accuracy `0.8424`, nhưng Prediabetes recall là `0.0000` và Diabetes recall là `0.0000`, nên accuracy cao chủ yếu do majority class và không phải bằng chứng hữu ích về khả năng phát hiện minority class.

Với giao thức Conv1D đã đóng băng, BasicCNN1D có test macro F1 cao nhất: `0.4392`. Accuracy của nó là `0.6953`, No diabetes recall là `0.7327`, Prediabetes recall là `0.2518`, và Diabetes recall là `0.5273`. Accuracy thấp hơn majority baseline là điều có thể hiểu được vì class weights cân bằng đánh đổi majority-class recall để tăng minority-class recall.

AlexNetInspired1D có Prediabetes recall cao nhất, `0.5094`, nhưng Diabetes recall chỉ `0.3412` và macro F1 là `0.4075`. VGGInspired1D có Diabetes recall cao nhất, `0.6777`, nhưng Prediabetes recall giảm còn `0.1065` và macro F1 là `0.4261`. ResNetInspired1D có raw accuracy cao nhất trong các CNN, `0.7124`, nhưng Prediabetes recall chỉ `0.0921`, nên accuracy đó không nên được xem là hiệu năng tổng thể mạnh.

Các kiến trúc sâu hơn không tạo cải thiện tổng thể rõ ràng. BasicCNN1D dùng 2,787 trainable parameters và huấn luyện trong 6.47 giây, trong khi AlexNetInspired1D, VGGInspired1D, và ResNetInspired1D dùng nhiều tham số hơn nhưng không vượt BasicCNN1D về macro F1 dưới protocol đóng băng.

Giới hạn tabular vẫn là điểm trung tâm: 21 vị trí feature đứng cạnh nhau vì thứ tự cột CSV, không phải vì chúng tạo thành chuỗi không gian hay thời gian tự nhiên. Các kết quả này đánh giá adaptation Conv1D theo yêu cầu assignment; chúng không cho thấy CNN là tối ưu cho bài toán diabetes tabular.
